1. Environment & imports (clean, portable)

In [ ]:
# === Environment / imports ===
import sys, os, warnings
print(f"Python version: {sys.version}")

# GPU check (no nvidia-smi dependency; works on Kaggle & Colab)
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Core scientific stack
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# ML utilities
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
try:
    import umap
except ImportError:
    !pip install -q umap-learn
    import umap

# Biology
try:
    from Bio import SeqIO
    from Bio.Seq import Seq
    from Bio.SeqRecord import SeqRecord
except ImportError:
    !pip install -q biopython
    from Bio import SeqIO
    from Bio.Seq import Seq
    from Bio.SeqRecord import SeqRecord

# PyTorch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("✓ Environment ready")


2. Synthetic genomic dataset (same idea, slightly safer defaults)

In [ ]:
# === DNA encoding utilities ===

class DNAEncoder:
    BASE_TO_IDX = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    IDX_TO_BASE = {0: 'A', 1: 'C', 2: 'G', 3: 'T'}

    @staticmethod
    def one_hot_encode(sequence: str) -> np.ndarray:
        seq_upper = sequence.upper()
        encoded = np.zeros((4, len(seq_upper)), dtype=np.float32)
        for idx, nucleotide in enumerate(seq_upper):
            if nucleotide in DNAEncoder.BASE_TO_IDX:
                encoded[DNAEncoder.BASE_TO_IDX[nucleotide], idx] = 1.0
        return encoded

    @staticmethod
    def decode_one_hot(encoded_array: np.ndarray) -> str:
        sequence = []
        for i in range(encoded_array.shape[1]):
            col = encoded_array[:, i]
            if np.max(col) < 0.5:
                sequence.append('N')
            else:
                base_idx = np.argmax(col)
                sequence.append(DNAEncoder.IDX_TO_BASE[base_idx])
        return ''.join(sequence)

    @staticmethod
    def compute_gc_content(sequence: str) -> float:
        seq_upper = sequence.upper()
        gc_count = seq_upper.count('G') + seq_upper.count('C')
        return (gc_count / len(seq_upper)) * 100 if len(seq_upper) > 0 else 0.0


# Quick sanity test
test_seq = "ATCGATCGATCG"
encoded = DNAEncoder.one_hot_encode(test_seq)
decoded = DNAEncoder.decode_one_hot(encoded)
print("Test DNA encoder:")
print(" original:", test_seq)
print(" decoded :", decoded)
print(" shape   :", encoded.shape)


# === Synthetic genome generator ===

def create_synthetic_genome(
    length=2_000_000,
    output_file="synthetic_genome.fasta",
    gc_content=0.36,
    seed=42,
):
    np.random.seed(seed)

    gc_prob = gc_content / 2
    at_prob = (1 - gc_content) / 2
    bases = ['A', 'T', 'G', 'C']
    weights = [at_prob, at_prob, gc_prob, gc_prob]

    print(f"Generating {length/1e6:.1f} Mb synthetic genome...")
    sequence = ''.join(np.random.choice(bases, size=length, p=weights))

    actual_gc = DNAEncoder.compute_gc_content(sequence)
    record = SeqRecord(
        Seq(sequence),
        id="synthetic_chromosome",
        description=f"Synthetic {length/1e6:.1f}Mb genome | Target GC={gc_content:.1%}",
    )
    SeqIO.write(record, output_file, "fasta")

    print(f"✓ Genome created: {output_file}")
    print(f" Actual GC content: {actual_gc:.2f}%")
    return output_file

genome_file = create_synthetic_genome(
    length=2_000_000,  # smaller default for Colab; concept unchanged
    gc_content=0.36,
)


In [ ]:
# === Genomic Dataset ===

class GenomicDataset(Dataset):
    def __init__(
        self,
        fasta_file,
        window_size=1024,
        stride=512,
        max_samples=20_000,
        filter_n_threshold=0.1,
    ):
        self.window_size = window_size
        self.sequences = []

        print(f"Loading sequences from {fasta_file}...")
        for record in SeqIO.parse(fasta_file, "fasta"):
            sequence = str(record.seq).upper()
            for i in range(0, len(sequence) - window_size + 1, stride):
                if max_samples and len(self.sequences) >= max_samples:
                    break
                chunk = sequence[i : i + window_size]
                n_prop = chunk.count('N') / len(chunk)
                if n_prop <= filter_n_threshold:
                    self.sequences.append(chunk)
            if max_samples and len(self.sequences) >= max_samples:
                break

        overlap = window_size - stride
        print("✓ Dataset created:")
        print(f" Sequences: {len(self.sequences):,}")
        print(f" Window: {window_size} bp")
        print(f" Stride: {stride} bp")
        print(f" Overlap: {overlap} bp ({overlap/window_size*100:.1f}%)")

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = self.sequences[idx]
        encoded = DNAEncoder.one_hot_encode(sequence)   # (4, L)
        encoded_flat = encoded.flatten()                # (4*L,)
        return torch.tensor(encoded_flat, dtype=torch.float32)

    def get_sequence(self, idx):
        return self.sequences[idx]


dataset = GenomicDataset(
    fasta_file=genome_file,
    window_size=1024,
    stride=512,
    max_samples=20_000,  # smaller default; adjust up if GPU is strong
)

print("Sample check:")
print(" tensor shape:", dataset[0].shape)
print(" sample seq  :", dataset.get_sequence(0)[:60], "...")


3. Hierarchical VAE model (same architecture, syntax fixed)


In [ ]:
# === Hierarchical VAE ===

class HierarchicalVAE(nn.Module):
    """
    Multi-scale VAE with three latent levels.
    Input (4096) -> encoder -> z1(256), z2(512), z3(1024)
    Concatenated (1792) -> decoder -> reconstruction (4096)
    """

    def __init__(self, input_dim=4096, latent_dims=None, dropout=0.3):
        super().__init__()
        if latent_dims is None:
            latent_dims = [256, 512, 1024]

        self.input_dim = input_dim
        self.latent_dims = latent_dims

        # Encoder
        self.enc1 = nn.Sequential(
            nn.Linear(input_dim, 2048),
            nn.LayerNorm(2048),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.enc2 = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.enc3 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Latent projections
        self.z1_mu = nn.Linear(512, latent_dims[0])
        self.z1_logvar = nn.Linear(512, latent_dims[0])

        self.z2_mu = nn.Linear(1024, latent_dims[1])
        self.z2_logvar = nn.Linear(1024, latent_dims[1])

        self.z3_mu = nn.Linear(2048, latent_dims[2])
        self.z3_logvar = nn.Linear(2048, latent_dims[2])

        # Decoder
        total_latent_dim = sum(latent_dims)
        self.dec1 = nn.Sequential(
            nn.Linear(total_latent_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.dec2 = nn.Sequential(
            nn.Linear(512, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.dec3 = nn.Sequential(
            nn.Linear(1024, 2048),
            nn.LayerNorm(2048),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.output = nn.Linear(2048, input_dim)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def encode(self, x):
        h1 = self.enc1(x)
        h2 = self.enc2(h1)
        h3 = self.enc3(h2)

        z1_mu = self.z1_mu(h3)
        z1_logvar = self.z1_logvar(h3)
        z1 = self.reparameterize(z1_mu, z1_logvar)

        z2_mu = self.z2_mu(h2)
        z2_logvar = self.z2_logvar(h2)
        z2 = self.reparameterize(z2_mu, z2_logvar)

        z3_mu = self.z3_mu(h1)
        z3_logvar = self.z3_logvar(h1)
        z3 = self.reparameterize(z3_mu, z3_logvar)

        latents = (z1, z2, z3)
        params = [(z1_mu, z1_logvar), (z2_mu, z2_logvar), (z3_mu, z3_logvar)]
        return latents, params

    def decode(self, latents):
        z = torch.cat(latents, dim=-1)
        h = self.dec1(z)
        h = self.dec2(h)
        h = self.dec3(h)
        return self.output(h)

    def forward(self, x):
        latents, params = self.encode(x)
        recon = self.decode(latents)
        return recon, latents, params

    def sample(self, num_samples, device=None):
        if device is None:
            device = next(self.parameters()).device
        self.eval()
        with torch.no_grad():
            z1 = torch.randn(num_samples, self.latent_dims[0], device=device)
            z2 = torch.randn(num_samples, self.latent_dims[1], device=device)
            z3 = torch.randn(num_samples, self.latent_dims[2], device=device)
            latents = (z1, z2, z3)
            samples = self.decode(latents)
        return samples


model = HierarchicalVAE(
    input_dim=1024 * 4,
    latent_dims=[256, 512, 1024],
    dropout=0.3,
)

total_params = sum(p.numel() for p in model.parameters())
print("MODEL:")
print(" latent dims:", model.latent_dims)
print(" total params:", f"{total_params:,}")
print(" size ~", f"{total_params * 4 / 1e6:.1f} MB (fp32)")


4. Loss, loaders, and training loop (simplified but same logic)


In [ ]:
# === VAE loss with β-annealing ===

def vae_loss(recon_x, x, latent_params, beta=1.0, kl_weights=None):
    if kl_weights is None:
        kl_weights = [1.0, 1.0, 1.0]

    recon_loss = F.mse_loss(recon_x, x, reduction="sum") / x.size(0)

    kl_per_level = []
    kl_loss = 0.0
    for w, (mu, logvar) in zip(kl_weights, latent_params):
        kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=-1)
        kl = kl.mean()
        kl_per_level.append(kl.item())
        kl_loss = kl_loss + w * kl

    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss, kl_per_level


def beta_schedule(epoch, warmup_epochs=15, max_beta=1.0, mode="linear"):
    if mode == "constant":
        return max_beta
    elif mode == "linear":
        if epoch < warmup_epochs:
            return (epoch / warmup_epochs) * max_beta
        return max_beta
    elif mode == "cosine":
        import math
        if epoch < warmup_epochs:
            progress = epoch / warmup_epochs
            return max_beta * (1 - math.cos(progress * math.pi)) / 2
        return max_beta
    return max_beta


In [ ]:
# === Data loaders ===

device = "cuda" if torch.cuda.is_available() else "cpu"

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42),
)

batch_size = 64 if torch.cuda.is_available() else 32
num_workers = 2 if torch.cuda.is_available() else 0

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

print("DATA:")
print(" train:", len(train_dataset))
print(" val  :", len(val_dataset))
print(" test :", len(test_dataset))
print(" batch size:", batch_size)


In [ ]:
# === Training loop ===

def train_model(model, train_loader, val_loader, epochs=20, lr=1e-3, device="cuda"):
    model.to(device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-5,
        betas=(0.9, 0.999),
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
        verbose=True,
        min_lr=1e-6,
    )

    history = {
        "train_loss": [],
        "train_recon": [],
        "train_kl": [],
        "val_loss": [],
        "val_recon": [],
        "val_kl": [],
        "kl_level1": [],
        "kl_level2": [],
        "kl_level3": [],
        "beta_values": [],
        "learning_rates": [],
    }

    best_val = float("inf")
    patience = 10
    patience_counter = 0

    for epoch in range(epochs):
        beta = beta_schedule(epoch, warmup_epochs=15, mode="linear")
        history["beta_values"].append(beta)
        history["learning_rates"].append(optimizer.param_groups[0]["lr"])

        # Train
        model.train()
        train_loss = train_recon = train_kl = 0.0
        kl_levels = [0.0, 0.0, 0.0]

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
        for batch in pbar:
            x = batch.to(device)

            optimizer.zero_grad()
            recon, latents, params = model(x)
            loss, recon_loss, kl_loss, kl_per_level = vae_loss(
                recon, x, params, beta=beta
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            train_recon += recon_loss.item()
            train_kl += kl_loss.item()
            for i in range(3):
                kl_levels[i] += kl_per_level[i]

            pbar.set_postfix(
                loss=f"{loss.item():.3f}",
                recon=f"{recon_loss.item():.3f}",
                kl=f"{kl_loss.item():.2f}",
                beta=f"{beta:.2f}",
            )

        n_train = len(train_loader)
        avg_train_loss = train_loss / n_train
        avg_train_recon = train_recon / n_train
        avg_train_kl = train_kl / n_train
        avg_kl_levels = [kl / n_train for kl in kl_levels]

        # Validation
        model.eval()
        val_loss = val_recon = val_kl = 0.0
        with torch.no_grad():
            for batch in val_loader:
                x = batch.to(device)
                recon, latents, params = model(x)
                loss, recon_loss, kl_loss, _ = vae_loss(recon, x, params, beta=beta)
                val_loss += loss.item()
                val_recon += recon_loss.item()
                val_kl += kl_loss.item()

        n_val = len(val_loader)
        avg_val_loss = val_loss / n_val
        avg_val_recon = val_recon / n_val
        avg_val_kl = val_kl / n_val

        history["train_loss"].append(avg_train_loss)
        history["train_recon"].append(avg_train_recon)
        history["train_kl"].append(avg_train_kl)
        history["val_loss"].append(avg_val_loss)
        history["val_recon"].append(avg_val_recon)
        history["val_kl"].append(avg_val_kl)
        history["kl_level1"].append(avg_kl_levels[0])
        history["kl_level2"].append(avg_kl_levels[1])
        history["kl_level3"].append(avg_kl_levels[2])

        scheduler.step(avg_val_loss)

        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f" Train: L={avg_train_loss:.4f} | R={avg_train_recon:.4f} | KL={avg_train_kl:.4f}")
        print(f" Val  : L={avg_val_loss:.4f} | R={avg_val_recon:.4f} | KL={avg_val_kl:.4f}")
        print(
            " KL levels:",
            f"L1={avg_kl_levels[0]:.2f}",
            f"L2={avg_kl_levels[1]:.2f}",
            f"L3={avg_kl_levels[2]:.2f}",
        )
        print(f" LR={optimizer.param_groups[0]['lr']:.2e} | beta={beta:.3f}")

        if avg_val_loss < best_val:
            best_val = avg_val_loss
            patience_counter = 0
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "history": history,
                },
                "best_model.pth",
            )
            print(f" ✓ Best model saved (val={best_val:.4f})")
        else:
            patience_counter += 1
            print(f" Patience {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("Early stopping.")
                break

    return history


print("Starting training on", device)
history = train_model(model, train_loader, val_loader, epochs=20, lr=1e-3, device=device)
